# ICV P03: Kaggle baseline v3, ResNet-18, seed 43

1. Import this notebook into Kaggle and keep it **private**.
2. Set **GPU T4 x2** and **Internet on**.
3. Use **Save Version > Save & Run All** for a fresh run.
4. Download **icv_baseline_v3_results.zip** from Output when the last cell says Finished.

This runs only the ImageNet-pretrained ResNet-18 baseline with basic preprocessing: seed 43, at most 50 epochs, early stopping patience 10. It uses the fixed NEU-CLS v2 manifest and evaluates only validation, not test. Settings are not changed to obtain a particular score.

**Fix for the previous failure:** there is no editable package installation. The pinned repository is imported directly through src and PYTHONPATH, so Kaggle Python 3.13 can reach the compatibility checks rather than being rejected by packaging metadata. The pinned training revision declares Python 3.11-3.12; the current project also accepts Python 3.13. Source compatibility is checked through imports and the repository tests before training; it is not assumed.

Kaggle's existing CUDA torch/torchvision pair is retained. Runtime versions and this deviation from uv.lock are recorded. Failed setup, data checks or tests block later training/evaluation cells even if Run All continues after an exception. No credentials or precomputed results are embedded. Raw data stays under /kaggle/temp and is excluded from the results ZIP.

**RAR extraction fix:** only unar is used. The presence of 7z or unrar does not skip its installation. The official downloader downloads the archive without extracting it; unar then extracts into a separate empty v3 directory. No fallback decoder is allowed to reuse partially extracted files, and the command has no interactive input. All image hashes are verified afterwards.

unar command options follow the [Debian manual](https://manpages.debian.org/unstable/unar/unar.1.en.html).

## 1. Prepare the pinned source and verify the GPU environment

This cell installs only missing/incompatible non-GPU dependencies. It does not install the project package or replace torch/torchvision.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import tomllib
from datetime import UTC, datetime
from pathlib import Path

from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet

SETUP_OK = False
DATA_OK = False
TESTS_OK = False
TRAINING_OK = False
VALIDATION_OK = False
checkpoint = None

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Kaggle-only notebook: local execution and training are disabled.")
if sys.version_info < (3, 11):  # noqa: UP036 -- validate the actual Kaggle runtime
    raise RuntimeError("Python 3.11 or newer is required.")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MPLCONFIGDIR"] = "/kaggle/temp/icv_mpl_v3"

REPO_URL = "https://github.com/kujifined/Introduction-to-cv.git"
REPO_COMMIT = "f25c6142dfa6088d5da70d5a590f65fef08d76b8"
EXPECTED_MANIFEST_SHA256 = "df784b6590f90947db6fd40c8d2e9650ab168744a0bd400e13dc535927f8232d"
EXPECTED_ARCHIVE_SHA256 = "a85bcc6bfa1a40e76f28079e135694aaf5f6a4ee01203f7eaacff0ebafaa96f8"
PROJECT_ROOT = Path("/kaggle/working/icv_repo_v3")
ARTIFACT_ROOT = Path("/kaggle/working/icv_baseline_v3")
TEMP_DATA = Path("/kaggle/temp/icv_neu_cls_v3")
SEED = 43
RUN_SUFFIX = "kaggle_baseline_v3_20261013"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


def run(command, *, cwd=PROJECT_ROOT):
    print("$", " ".join(str(x) for x in command), flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


if not PROJECT_ROOT.exists():
    run(["git", "clone", REPO_URL, PROJECT_ROOT], cwd="/kaggle/working")
    run(["git", "checkout", "--detach", REPO_COMMIT])
if not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError("Project directory is occupied by something other than the checkout.")
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()
if revision != REPO_COMMIT:
    raise RuntimeError("Unexpected checkout revision. Start a fresh Kaggle run.")
run(["git", "diff", "--exit-code", "HEAD"])
os.chdir(PROJECT_ROOT)

# No pip install -e: the old package metadata excludes Python 3.13.
source_dir = str(PROJECT_ROOT / "src")
sys.path.insert(0, source_dir)
previous_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    source_dir + os.pathsep + previous_pythonpath if previous_pythonpath else source_dir
)

project_metadata = tomllib.loads((PROJECT_ROOT / "pyproject.toml").read_text())
requirements = [Requirement(value) for value in project_metadata["project"]["dependencies"]]
requirements.extend([Requirement("gdown>=5.2,<6"), Requirement("pytest>=8.2,<10")])
gpu_names = {"torch", "torchvision"}


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


gpu_versions_before = {name: installed_version(name) for name in gpu_names}
missing_non_gpu = []
for requirement in requirements:
    if requirement.marker is not None and not requirement.marker.evaluate():
        continue
    actual = installed_version(requirement.name)
    compatible = actual is not None and actual in requirement.specifier
    if requirement.name in gpu_names:
        if not compatible:
            raise RuntimeError(
                f"Kaggle GPU package {requirement.name}={actual} does not meet "
                f"{requirement.specifier}. No GPU package will be replaced automatically."
            )
    elif not compatible:
        missing_non_gpu.append(str(requirement))
if missing_non_gpu:
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--only-binary=:all:",
            *missing_non_gpu,
        ]
    )
for requirement in requirements:
    if requirement.marker is not None and not requirement.marker.evaluate():
        continue
    actual = installed_version(requirement.name)
    if actual is None or actual not in requirement.specifier:
        raise RuntimeError(f"Dependency check failed: {requirement}; installed={actual}")
if gpu_versions_before != {name: installed_version(name) for name in gpu_names}:
    raise RuntimeError("GPU package versions unexpectedly changed.")

import torch  # noqa: E402 -- import after dependency and environment setup
import torchvision  # noqa: E402, F401 -- smoke-test the installed CUDA package pair

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle. CPU training is disabled.")
import industrial_surface_defects  # noqa: E402 -- pinned src path is configured above

expected_package = (PROJECT_ROOT / "src/industrial_surface_defects").resolve()
if Path(industrial_surface_defects.__file__).resolve().parent != expected_package:
    raise RuntimeError("A different project module is cached. Restart the Kaggle session.")

versions = {req.name: installed_version(req.name) for req in requirements}
python_version = ".".join(map(str, sys.version_info[:3]))
declared_range = project_metadata["project"]["requires-python"]
environment = {
    "recorded_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": revision,
    "python": python_version,
    "gpu_model": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "packages": versions,
    "project_declared_python_range": declared_range,
    "outside_declared_packaged_python_range": python_version not in SpecifierSet(declared_range),
    "execution_mode": "Pinned source imported via src/PYTHONPATH; no package installation.",
    "environment_note": (
        "Kaggle CUDA runtime retained; uv.lock not used. Tests must pass before training."
    ),
    "seed": SEED,
    "planned_max_epochs": 50,
    "test_evaluation_enabled": False,
}
save_json(ARTIFACT_ROOT / "cloud_environment.json", environment)
(ARTIFACT_ROOT / "pip_freeze.txt").write_text(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    ),
    encoding="utf-8",
)
SETUP_OK = True
print(json.dumps(environment, indent=2))
print("SETUP OK")

## 2. Verify the official dataset and run implementation tests

Known duplicate decisions and the fixed split are preserved. File hashes include the entire dataset for integrity checking; no test predictions or test metrics are computed.

In [ ]:
DATA_OK = False
TESTS_OK = False
if not globals().get("SETUP_OK", False):
    raise RuntimeError("Setup did not finish. Fix the first code cell before proceeding.")

import pandas as pd  # noqa: E402 -- setup must pass before importing dependencies

from industrial_surface_defects.data import verify_manifest_files  # noqa: E402

manifest = PROJECT_ROOT / "data/manifests/neu_cls_v2.csv"
assert sha256(manifest) == EXPECTED_MANIFEST_SHA256
run(["git", "diff", "--exit-code", "HEAD", "--", "data/manifests"])

# Download directly from the official source. Do not re-upload a dataset copy.
TEMP_DATA.mkdir(parents=True, exist_ok=True)
archive = TEMP_DATA / "NEU-CLS.rar"
raw_root = TEMP_DATA / "raw"


def require_unar():
    # Checking for unrar or 7z is not sufficient: they caused the previous failure.
    decoder = shutil.which("unar")
    if decoder is None:
        if shutil.which("apt-get") is None:
            raise RuntimeError("unar is required and apt-get is unavailable.")
        run(["apt-get", "-qq", "update"], cwd="/kaggle/working")
        run(
            [
                "apt-get",
                "-qq",
                "install",
                "-y",
                "--no-install-recommends",
                "unar",
            ],
            cwd="/kaggle/working",
        )
        decoder = shutil.which("unar")
    if decoder is None:
        raise RuntimeError("unar installation completed but its executable was not found.")
    return decoder


def extract_with_unar(decoder, archive_path, destination):
    destination = Path(destination)
    if destination.exists() and (not destination.is_dir() or any(destination.iterdir())):
        raise FileExistsError(
            "Extraction directory is occupied. Start a fresh Save & Run All; "
            "existing files will not be deleted or overwritten."
        )
    destination.mkdir(parents=True, exist_ok=True)
    command = [
        decoder,
        "-q",
        "-D",
        "-o",
        str(destination),
        str(archive_path),
    ]
    print("$", " ".join(command), flush=True)
    result = subprocess.run(
        command,
        cwd="/kaggle/working",
        stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=120,
        check=False,
    )
    (ARTIFACT_ROOT / "extraction.log").write_text(result.stdout, encoding="utf-8")
    if result.returncode != 0:
        raise RuntimeError(
            f"unar failed with exit code {result.returncode}: {result.stdout[-1500:]}"
        )
    if not (destination / "NEU-CLS").is_dir():
        raise RuntimeError("The expected NEU-CLS image directory was not extracted.")
    return command


decoder = require_unar()
if not archive.exists():
    # No --extract flag: bypass the old downloader's unsafe decoder fallback.
    run(
        [
            sys.executable,
            "scripts/download_neu_cls.py",
            "--acknowledge-source-terms",
            "--archive",
            archive,
        ]
    )
assert sha256(archive) == EXPECTED_ARCHIVE_SHA256
extraction_command = extract_with_unar(decoder, archive, raw_root)
save_json(
    ARTIFACT_ROOT / "extraction_metadata.json",
    {
        "decoder": decoder,
        "command": extraction_command,
        "archive_sha256": sha256(archive),
        "raw_data_in_results_zip": False,
    },
)

data_link = PROJECT_ROOT / "data/raw/NEU-CLS"
data_link.parent.mkdir(parents=True, exist_ok=True)
if not data_link.exists():
    data_link.symlink_to(raw_root, target_is_directory=True)
else:
    assert data_link.resolve() == raw_root.resolve(), "Unexpected existing dataset directory."

verify_manifest_files(manifest, data_link)
rows = pd.read_csv(manifest)
assert rows.groupby("split").size().to_dict() == {
    "excluded": 1,
    "test": 360,
    "train": 1259,
    "val": 180,
}
print(rows[rows.status == "ok"].groupby(["class_name", "split"]).size().unstack())
DATA_OK = True
save_json(
    ARTIFACT_ROOT / "dataset_verification.json",
    {
        "manifest_sha256": sha256(manifest),
        "archive_sha256": sha256(archive),
        "split_counts": rows.groupby("split").size().to_dict(),
        "every_manifest_file_hash_verified": True,
    },
)
print("Every image hash and the fixed split verified.")
# These are implementation tests; none trains a model on NEU-CLS.
pytest_command = [sys.executable, "-m", "pytest", "-q"]
print("$", " ".join(pytest_command), flush=True)
test_result = subprocess.run(
    pytest_command,
    cwd=PROJECT_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)
(ARTIFACT_ROOT / "pytest.log").write_text(test_result.stdout, encoding="utf-8")
print(test_result.stdout)
if test_result.returncode != 0:
    raise RuntimeError("Repository tests failed. See pytest.log; training remains blocked.")
TESTS_OK = True
print("DATA AND TESTS OK; training is now permitted.")

## 3. Train the single baseline

This is the GPU workload. It can run only after setup, dataset checks and all repository tests pass. Existing model-run directories are never overwritten.

In [ ]:
import contextlib
from dataclasses import replace

TRAINING_OK = False
VALIDATION_OK = False
checkpoint = None
if not all(globals().get(flag, False) for flag in ("SETUP_OK", "DATA_OK", "TESTS_OK")):
    raise RuntimeError("Setup, data verification and tests must all pass before training.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable; CPU training is disabled.")

from industrial_surface_defects.config import load_config  # noqa: E402
from industrial_surface_defects.train import train  # noqa: E402 -- setup/tests must pass

config = load_config(
    "configs/resnet18_minimal.yaml",
    seed=SEED,
    device="cuda",
)
config = replace(
    config,
    experiment=replace(config.experiment, output_root=str(ARTIFACT_ROOT / "outputs")),
    training=replace(config.training, num_workers=2),
)
assert config.model.pretrained
assert config.data.augmentation == "minimal"
assert config.training.epochs == 50
assert config.training.early_stopping_patience == 10
assert config.training.device == "cuda"
print("Starting the single ResNet-18 baseline; seed =", SEED)


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


with (
    (ARTIFACT_ROOT / "training.log").open("a", encoding="utf-8") as log_stream,
    contextlib.redirect_stdout(Tee(sys.stdout, log_stream)),
):
    checkpoint = train(config, run_suffix=RUN_SUFFIX)
TRAINING_OK = True
print("Completed training. Saved checkpoint:", checkpoint)

## 4. Evaluate validation and independently check the results

All 180 validation images are used. Accuracy and macro-F1 are also calculated without the project's metric function. Shuffled labels provide a chance-level sanity check.

In [ ]:
VALIDATION_OK = False
if not all(
    globals().get(flag, False) for flag in ("SETUP_OK", "DATA_OK", "TESTS_OK", "TRAINING_OK")
):
    raise RuntimeError("Training did not complete successfully. Validation is blocked.")
if checkpoint is None or not Path(checkpoint).is_file():
    raise RuntimeError("The current run has no saved checkpoint.")

import numpy as np  # noqa: E402 -- current training must succeed before evaluation

from industrial_surface_defects.evaluate import evaluate  # noqa: E402

evaluation_dir = evaluate(
    checkpoint,
    split="val",
    acknowledge_test_use=False,
    device_name="cuda",
    num_workers=2,
)
metrics = json.loads((evaluation_dir / "metrics.json").read_text())
predictions = pd.read_csv(evaluation_dir / "predictions.csv")
expected_validation = rows.loc[rows.split == "val"].set_index("path")
assert len(predictions) == 180
assert not predictions.path.duplicated().any()
assert set(predictions.path) == set(expected_validation.index)
assert set(predictions.path).isdisjoint(set(rows.loc[rows.split == "train", "path"]))
assert set(predictions.path).isdisjoint(set(rows.loc[rows.split == "test", "path"]))
assert np.array_equal(
    predictions.true_index.to_numpy(),
    expected_validation.loc[predictions.path, "class_index"].to_numpy(),
)
# Independent arithmetic check, without the project's metric function.
y = predictions.true_index.to_numpy()
pred = predictions.predicted_index.to_numpy()
manual_accuracy = float(np.mean(y == pred))
manual_f1 = []
for cls in range(6):
    tp = int(np.sum((y == cls) & (pred == cls)))
    fp = int(np.sum((y != cls) & (pred == cls)))
    fn = int(np.sum((y == cls) & (pred != cls)))
    manual_f1.append(2 * tp / (2 * tp + fp + fn))
manual_macro_f1 = float(np.mean(manual_f1))
assert abs(manual_accuracy - metrics["accuracy"]) < 1e-12
assert abs(manual_macro_f1 - metrics["macro_f1"]) < 1e-12
rng = np.random.default_rng(20261013)
shuffled_accuracy = float(np.mean([np.mean(pred == rng.permutation(y)) for _ in range(1000)]))
audit = {
    "validation_images": len(predictions),
    "correct_predictions": int(np.sum(y == pred)),
    "manual_accuracy": manual_accuracy,
    "manual_macro_f1": manual_macro_f1,
    "shuffled_label_mean_accuracy_1000_checks": shuffled_accuracy,
    "expected_six_class_chance_accuracy": 1 / 6,
    "manifest_sha256": sha256(manifest),
    "checkpoint_sha256": sha256(checkpoint),
    "train_validation_paths_disjoint": True,
    "test_evaluation_performed": False,
}
(ARTIFACT_ROOT / "independent_validation_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
print("VALIDATION METRICS")
print(json.dumps(metrics, indent=2))
print("INDEPENDENT CHECK")
print(json.dumps(audit, indent=2))
print(pd.read_csv(evaluation_dir / "per_class_metrics.csv").to_string(index=False))

from IPython.display import display  # noqa: E402 -- figures are shown after the audit
from PIL import Image  # noqa: E402

display(Image.open(checkpoint.parent / "learning_curves.png"))
display(Image.open(evaluation_dir / "confusion_matrix.png"))
VALIDATION_OK = True
print("VALIDATION OK; the next cell packages the results.")
print("No test predictions, test metrics or test corruptions were generated.")

## 5. Package the results

The ZIP contains only generated experiment evidence and the checkpoint, not raw data. Download it through Kaggle Output and share it for the report.

In [ ]:
if not globals().get("VALIDATION_OK", False):
    raise RuntimeError("Validation/audit did not finish. No completed results ZIP will be created.")
results_zip_base = Path("/kaggle/working/icv_baseline_v3_results")
if results_zip_base.with_suffix(".zip").exists():
    raise FileExistsError("Results ZIP already exists; preserve it and use a fresh saved run.")
# Package only ARTIFACT_ROOT, not the checkout, archive or raw images.
results_zip = shutil.make_archive(str(results_zip_base), "zip", root_dir=ARTIFACT_ROOT)
print("Finished. Download:", results_zip)
print("Selected checkpoint:", checkpoint)
print("Validation metrics:", evaluation_dir / "metrics.json")
print("Training history:", checkpoint.parent / "history.csv")
print("Runtime and packages:", ARTIFACT_ROOT / "cloud_environment.json")
print("The ZIP includes checkpoint, logs, history, predictions, figures and audit.")
print("No test evaluation or corruption tests were run.")